# 05 — Governance: Tags, Comments, Column Masks, Lineage

## Why this matters — Well-Architected Framework

| WAF pillar | What this notebook does, and why it's good |
|------------|---------------------------------------------|
| **Data Governance** | Catalog-wide comments + domain tags mean any asset carrying `mlb_analytics=''` auto-appears in the Databricks One Domain. Governance-as-data, not governance-as-spreadsheet. |
| **Security, Compliance & Privacy** | `contains_pii` column tags make PII **queryable** — you can answer "which tables touch PII?" with a single SQL query instead of a data-map project. The `attendance` column mask is the real-deal enforcement: non-owners literally cannot see the value. |
| **Reliability** | Runtime lineage from `system.access.table_lineage` is automatic — no manifest files, no orchestrator metadata to keep in sync. If it ran, it's in the lineage graph. |
| **Operational Excellence** | The governance here is entirely declarative — every `COMMENT ON`, every `SET TAGS`, every `SET MASK` is idempotent SQL you can check into git and re-apply anywhere. |

> **Presenter tip:** show the Catalog Explorer ERD *live* after running this
> notebook. The PKs/FKs from 03 + 04 plus the comments here produce a
> diagram that looks like a BI team's dream — completely free, no tool to buy.


In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
DATA_OWNER    = os.getenv("WAF_DATA_OWNER", "mlb_analytics_team")

S = f"{UC_CATALOG}.{SILVER_SCHEMA}"
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print(f"Silver: {S}")
print(f"Gold:   {G}")


Silver: alexander_booth.mlb_gumbo_waf_silver
Gold:   alexander_booth.mlb_gumbo_waf_gold


## Table- and column-level comments

Comments surface in Catalog Explorer, Databricks One, AI/BI dashboards,
and Genie — good comments materially improve Genie's SQL accuracy.

In [2]:
table_comments = {
    f"{G}.fact_games":          "One row per MLB game. Use this fact table for game-grain analyses (attendance, outcomes, flags).",
    f"{G}.fact_pitches":        "One row per pitch event. Use this fact table for pitch-grain analyses (velocity, spin, pitch mix, strike rate).",
    f"{G}.dim_team":            "MLB team dimension. Joined to fact_games via home_team_sk / away_team_sk.",
    f"{G}.dim_pitcher":         "Pitcher dimension including throwing hand.",
    f"{G}.dim_batter":          "Batter dimension including stance.",
    f"{G}.dim_venue":           "Stadium / venue dimension.",
    f"{G}.dim_date":            "Standard date dimension covering 2020–2030.",
    f"{G}.v_pitcher_leaderboard":     "Season-to-date pitcher leaderboard — pitches thrown, avg velocity, strike %. Pitchers with >200 pitches only.",
    f"{G}.v_pitch_mix_by_type":       "Pitch-type aggregates — volume, velocity, break, spin by pitch type per season.",
    f"{G}.v_team_pitching_summary":   "Team-level pitching summary by season.",
}
for t, c in table_comments.items():
    spark.sql(f"COMMENT ON TABLE {t} IS '{c}'")
    print(f"  {t}")
print(f"\n{len(table_comments)} table comments applied.")


  alexander_booth.mlb_gumbo_waf_gold.fact_games


  alexander_booth.mlb_gumbo_waf_gold.fact_pitches


  alexander_booth.mlb_gumbo_waf_gold.dim_team


  alexander_booth.mlb_gumbo_waf_gold.dim_pitcher


  alexander_booth.mlb_gumbo_waf_gold.dim_batter


  alexander_booth.mlb_gumbo_waf_gold.dim_venue


  alexander_booth.mlb_gumbo_waf_gold.dim_date


  alexander_booth.mlb_gumbo_waf_gold.v_pitcher_leaderboard


  alexander_booth.mlb_gumbo_waf_gold.v_pitch_mix_by_type


  alexander_booth.mlb_gumbo_waf_gold.v_team_pitching_summary

10 table comments applied.


## Tags — domain / tier / owner

The `mlb_analytics` tag key is the spine of the Databricks One Domain for this
workspace. Whichever assets carry that tag key light up in the domain.

In [3]:
def set_tag_defensive(table, key, value):
    try:
        spark.sql(f"ALTER TABLE {table} SET TAGS ('{key}' = '{value}')")
        return True
    except Exception as e:
        msg = str(e)
        if "INVALID_PARAMETER_VALUE" in msg or "not an allowed value" in msg or "tag policy" in msg.lower():
            print(f"    skipped {table} {key}={value} (governed tag policy)")
        else:
            print(f"    skipped {table} {key}={value} ({msg[:120]})")
        return False


# Gold: tier=gold, owner, plus the bare mlb_analytics tag key for domain membership
for t in ["dim_date","dim_team","dim_pitcher","dim_batter","dim_venue","fact_games","fact_pitches"]:
    print(f"  tagging {G}.{t}")
    set_tag_defensive(f"{G}.{t}", "mlb_analytics", "")
    set_tag_defensive(f"{G}.{t}", "tier", "gold")
    set_tag_defensive(f"{G}.{t}", "data_owner", DATA_OWNER)


  tagging alexander_booth.mlb_gumbo_waf_gold.dim_date


  tagging alexander_booth.mlb_gumbo_waf_gold.dim_team


  tagging alexander_booth.mlb_gumbo_waf_gold.dim_pitcher


  tagging alexander_booth.mlb_gumbo_waf_gold.dim_batter


  tagging alexander_booth.mlb_gumbo_waf_gold.dim_venue


  tagging alexander_booth.mlb_gumbo_waf_gold.fact_games


  tagging alexander_booth.mlb_gumbo_waf_gold.fact_pitches


## Column tags — PII / sensitivity

GUMBO is public data, but we still tag pitcher/batter names as PII-ish to
demonstrate the pattern customers will use on real rosters + scouting reports.

In [4]:
pii_columns = [
    (f"{S}.pitch_data", "pitcher_name"),
    (f"{S}.pitch_data", "batter_name"),
    (f"{G}.dim_pitcher", "pitcher_name"),
    (f"{G}.dim_batter",  "batter_name"),
]
for tbl, col in pii_columns:
    try:
        spark.sql(f"ALTER TABLE {tbl} ALTER COLUMN {col} SET TAGS ('contains_pii' = 'y')")
        print(f"  tagged PII: {tbl}.{col}")
    except Exception as e:
        msg = str(e)
        if "INVALID_PARAMETER_VALUE" in msg or "not an allowed value" in msg or "tag policy" in msg.lower():
            print(f"    skipped PII {tbl}.{col} (governed tag policy)")
        else:
            print(f"    skipped PII {tbl}.{col} ({msg[:120]})")


  tagged PII: alexander_booth.mlb_gumbo_waf_silver.pitch_data.pitcher_name


  tagged PII: alexander_booth.mlb_gumbo_waf_silver.pitch_data.batter_name


  tagged PII: alexander_booth.mlb_gumbo_waf_gold.dim_pitcher.pitcher_name


  tagged PII: alexander_booth.mlb_gumbo_waf_gold.dim_batter.batter_name


## Column mask — redact `attendance` unless you're in the owner group

Shows the full column-mask pattern: a UDF + `ALTER TABLE ... ALTER COLUMN ... SET MASK`.
The cells below create the function and wire it up. If you don't have a group
named `{DATA_OWNER}` you can change the group name or skip this cell.

In [5]:
mask_func = f"{G}.mask_attendance"

# Step 1 — a SQL UDF that returns the real value only for owners. Everyone else gets NULL.
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {mask_func}(attendance INT)
    RETURNS INT
    RETURN CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('{DATA_OWNER}') THEN attendance
        ELSE NULL
    END
""")
print(f"Created masking function: {mask_func}")

# Step 2 — apply the mask. If the mask already exists this is a no-op.
try:
    spark.sql(f"ALTER TABLE {G}.fact_games ALTER COLUMN attendance SET MASK {mask_func}")
    print(f"  Applied mask on {G}.fact_games.attendance")
except Exception as e:
    msg = str(e)
    if "already has a column mask" in msg or "ALREADY" in msg:
        print("  Mask already applied — skipping.")
    else:
        print(f"  Skipping mask ({msg[:120]})")


Created masking function: alexander_booth.mlb_gumbo_waf_gold.mask_attendance


  Applied mask on alexander_booth.mlb_gumbo_waf_gold.fact_games.attendance


## Verify

In [6]:
print("Table tags on demo schemas:")
spark.sql(f"""
    SELECT schema_name, table_name, tag_name, tag_value
    FROM {UC_CATALOG}.information_schema.table_tags
    WHERE schema_name LIKE '{UC_SCHEMA}%'
    ORDER BY schema_name, table_name, tag_name
""").show(100, truncate=False)

print("\nColumn tags (PII):")
spark.sql(f"""
    SELECT schema_name, table_name, column_name, tag_name, tag_value
    FROM {UC_CATALOG}.information_schema.column_tags
    WHERE schema_name LIKE '{UC_SCHEMA}%'
""").show(50, truncate=False)

print("\nLineage (bronze -> silver -> gold):")
spark.sql(f"""
    SELECT source_table_full_name, target_table_full_name, event_time
    FROM system.access.table_lineage
    WHERE target_table_full_name LIKE '{UC_CATALOG}.{UC_SCHEMA}_%'
    ORDER BY event_time DESC
    LIMIT 20
""").show(truncate=False)


Table tags on demo schemas:


+------------------+------------+-------------+------------------+
|schema_name       |table_name  |tag_name     |tag_value         |
+------------------+------------+-------------+------------------+
|mlb_gumbo_waf_gold|dim_batter  |data_owner   |mlb_analytics_team|
|mlb_gumbo_waf_gold|dim_batter  |mlb_analytics|                  |
|mlb_gumbo_waf_gold|dim_batter  |tier         |gold              |
|mlb_gumbo_waf_gold|dim_date    |data_owner   |mlb_analytics_team|
|mlb_gumbo_waf_gold|dim_date    |mlb_analytics|                  |
|mlb_gumbo_waf_gold|dim_date    |tier         |gold              |
|mlb_gumbo_waf_gold|dim_pitcher |data_owner   |mlb_analytics_team|
|mlb_gumbo_waf_gold|dim_pitcher |mlb_analytics|                  |
|mlb_gumbo_waf_gold|dim_pitcher |tier         |gold              |
|mlb_gumbo_waf_gold|dim_team    |data_owner   |mlb_analytics_team|
|mlb_gumbo_waf_gold|dim_team    |mlb_analytics|                  |
|mlb_gumbo_waf_gold|dim_team    |tier         |gold           

+--------------------+-----------+------------+------------+---------+
|schema_name         |table_name |column_name |tag_name    |tag_value|
+--------------------+-----------+------------+------------+---------+
|mlb_gumbo_waf_silver|pitch_data |batter_name |contains_pii|y        |
|mlb_gumbo_waf_silver|pitch_data |pitcher_name|contains_pii|y        |
|mlb_gumbo_waf_gold  |dim_pitcher|pitcher_name|contains_pii|y        |
|mlb_gumbo_waf_gold  |dim_batter |batter_name |contains_pii|y        |
+--------------------+-----------+------------+------------+---------+


Lineage (bronze -> silver -> gold):


+-----------------------------------------------+----------------------------------------------------------+-----------------------+
|source_table_full_name                         |target_table_full_name                                    |event_time             |
+-----------------------------------------------+----------------------------------------------------------+-----------------------+
|NULL                                           |alexander_booth.mlb_gumbo_waf_gold.dq_results             |2026-04-21 20:43:01.887|
|NULL                                           |alexander_booth.mlb_gumbo_waf_gold.dq_results             |2026-04-21 20:42:59.2  |
|NULL                                           |alexander_booth.mlb_gumbo_waf_gold.dq_results             |2026-04-21 20:42:55.909|
|NULL                                           |alexander_booth.mlb_gumbo_waf_gold.dq_results             |2026-04-21 20:42:52.531|
|NULL                                           |alexander_booth.mlb_